# Week 10 Post-Class: Build YOUR Hugging Face Pipeline

## Goal
Build a working Hugging Face pipeline for a personal text task. This is the workflow you'll use when you need NLP at work.

## Two parts
1. **Part A:** Build a pipeline for a chosen task on YOUR data
2. **Part B:** Fine-tune DistilBERT on a custom text classification problem

## Time
60-90 minutes total

## Continuity
Same patterns from `week10_live_session.ipynb` — applied to your data.

## Part A: Pre-trained pipeline on your data

Pick a task that's relevant to YOUR work or interests:
- Sentiment analysis on emails / reviews / tweets — `pipeline`
- NER on news articles — `pipeline`
- Zero-shot classification with custom labels — `pipeline`
- Summarization of long documents — `AutoModel` (seq2seq; pipeline shortcut removed in v5)
- Translation of multilingual content — `AutoModel` (seq2seq; pipeline shortcut removed in v5)

**Provide ~5 input examples** that are real to your work.

Goal: in under 30 lines of code, get useful output.

In [ ]:
from transformers import pipeline

# YOUR TURN: pick a task.
# Tasks with a one-line pipeline() shortcut (transformers v5):
#   task = pipeline('sentiment-analysis')
#   task = pipeline('zero-shot-classification')
#   task = pipeline('ner', aggregation_strategy='simple')
#   task = pipeline('fill-mask')
#   task = pipeline('text-generation', model='gpt2')
#
# Summarization / translation / extractive QA had their pipeline shortcuts REMOVED
# in transformers v5 — use the AutoModel pattern instead (see week10_live_session.ipynb
# and week10_pair_programming.ipynb Tasks 2-4). Quick summarization template:
#   from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#   tok = AutoTokenizer.from_pretrained('google-t5/t5-small')
#   m = AutoModelForSeq2SeqLM.from_pretrained('google-t5/t5-small')
#   ids = m.generate(**tok('summarize: ' + text, return_tensors='pt',
#                          truncation=True, max_length=512), max_new_tokens=60)
#   print(tok.decode(ids[0], skip_special_tokens=True))

# YOUR TURN: provide 5+ real examples from your work/interests
your_examples = [
    # 'Example 1...',
    # 'Example 2...',
]

# YOUR TURN: run the pipeline and inspect outputs
# for example in your_examples:
#     result = task(example)
#     print(f'Input: {example}\nResult: {result}\n')

### Reflection on Part A

1. **Did the pipeline produce useful output?** Specific examples.
2. **Where did it fail?** What inputs caused bad outputs?
3. **Would you trust this in production?** What guardrails would you need?
4. **How does this compare to manual rule-based NLP** (regex, keyword matching)?

## Part B: Fine-tune DistilBERT on your text classification problem

If you have labeled text data (or can scrape some), fine-tune DistilBERT on it.

**Suggested data sources:**
- Your own emails categorized by folder
- Tweets you've categorized
- News articles by topic
- Movie reviews (IMDB available via `load_dataset('imdb')`)
- Yelp reviews (available via `load_dataset('yelp_review_full')`)
- Any HuggingFace dataset under https://huggingface.co/datasets

**Minimum:** 200 training examples per class. More is better.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from datasets import load_dataset, Dataset
import numpy as np

set_seed(42)

# YOUR TURN: load YOUR dataset
# Option A: from Hugging Face
# ds = load_dataset('imdb')
# train_ds = ds['train'].shuffle(seed=42).select(range(2000))
# test_ds = ds['test'].shuffle(seed=42).select(range(500))

# Option B: from your own CSV/list of texts and labels
# texts = [...]
# labels = [...]  # integers 0, 1, 2, ...
# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
# train_ds = Dataset.from_dict({'text': X_train, 'label': y_train})
# test_ds = Dataset.from_dict({'text': X_test, 'label': y_test})

# print(f'Training: {len(train_ds)} examples')

In [ ]:
# YOUR TURN: tokenize
# model_name = 'distilbert-base-uncased'
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# def tokenize(examples):
#     return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

# train_tokenized = train_ds.map(tokenize, batched=True)
# test_tokenized = test_ds.map(tokenize, batched=True)

In [ ]:
# YOUR TURN: load model + train
# num_labels = 2  # or however many classes you have
# model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# training_args = TrainingArguments(
#     output_dir='./my_classifier',
#     num_train_epochs=3,
#     per_device_train_batch_size=16,
#     learning_rate=5e-5,
#     eval_strategy='epoch',
#     logging_steps=50,
#     save_strategy='no',
#     report_to='none',
#     disable_tqdm=True,   # plain-text progress (avoids a transformers v5 notebook callback bug)
# )

# def compute_metrics(eval_pred):
#     predictions, labels = eval_pred
#     return {'accuracy': (np.argmax(predictions, axis=1) == labels).mean()}

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_tokenized,
#     eval_dataset=test_tokenized,
#     compute_metrics=compute_metrics,
# )

# trainer.train()
# print(trainer.evaluate())

# To run inference afterward, send inputs to the model's OWN device
# (works on CPU, CUDA, or Apple MPS):
#   device = next(model.parameters()).device
#   inputs = {k: v.to(device) for k, v in tokenizer(text, return_tensors='pt',
#             truncation=True, max_length=128).items()}
#   with torch.no_grad():
#       logits = model(**inputs).logits

## Reflection (write in `Week10_Self_Assessment.md`)

1. **What task did you build?** Briefly describe.
2. **What dataset did you use?** Source, size, classes.
3. **Final test accuracy:** ___________
4. **Did the model surprise you in any way?** Either positively or negatively.
5. **What would you have to do differently in 2015 to build this?** (Note: 2015 was pre-transformer.)
6. **What's the next thing you'd do to improve the model?**

## Looking ahead to Week 11

Today: encoder transformers for understanding. Next week: decoder transformers for generation. LLMs and diffusion models. Course finale.

**Pre-class for Week 11:** download Phi-3-mini OR Qwen2.5-0.5B from Hugging Face. Both work — pick one based on your machine's RAM. The pre-class email will have specific instructions.